<a href="https://colab.research.google.com/github/OSUrobotics/applevision_distance_bridge/blob/main/Week_1_Dynamics_2R_arm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

An excellent reference for Kinematics, Dynamics, and Control is *Modern Robotics* by Lynch and Park, which is available at https://hades.mech.northwestern.edu/index.php/Modern_Robotics.

*Acknowledgement*: ChatGPT was used to generate the section of code for the HTML-based robot simulation.

The general robot manipulator equation, often referred to as the dynamic model or equations of motion, is given by:

$M(\theta)\ddot{\theta} + c(\theta, \dot{\theta}) + g(\theta) = \tau$

Where:

*   $M(\theta)$ is the symmetric, positive-definite mass matrix
*   $c(\theta, \dot{\theta})$ is the vector of coriolis and centripetal torques
*   $g(\theta)$ is the vector of gravitational torques
*   $\tau$ is the vector of joint torques (or forces for prismatic joints)
*   $\theta$ is the vector of joint angles (or positions)
*   $\dot{\theta}$ is the vector of joint velocities
*   $\ddot{\theta}$ is the vector of joint accelerations

In this exercise we will simulate the forward dynamics of the simple, planar 2R arm. More specifically, given initial joint states (i.e. joint angles and joint velocities) and a known vector of joint torques, our goal is to simulate the robot's trajectory using

$\ddot{\theta} = M^{-1}(\theta)(\tau - c(\theta, \dot{\theta}) - g(\theta))$

In [ ]:
# Import required python libraries
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.animation as FuncAnimation

In [ ]:
# -----------------------------
# Simulation Parameters
# -----------------------------
dt = 0.02
T = 5
steps = int(T / dt)

time = np.linspace(0, T, steps)

In [ ]:
# -----------------------------
# Joint Torque Inputs
# -----------------------------
# Example shows time-varying torques
# Shape: (steps, 2)

tau = np.zeros((steps, 2))
tau[:, 0] = 0 * np.sin(1.0 * time)
tau[:, 1] = 0 * np.cos(1.5 * time)

In [ ]:
# -----------------------------
# Robot Parameters
# -----------------------------
l1 = 1.0   # Length of link 1 (m)
l2 = 1.0   # Length of link 2 (m)

m1 = 1.0   # Mass of link 1 (kg)
m2 = 1.0   # Mass of link 2 (kg)

g = 9.81   # Gravitational constant (m/s^2)

In [ ]:
# -----------------------------
# Initialize the State Variables
# -----------------------------
q = np.zeros((steps, 2))       # Joint angles (rad)
qd = np.zeros((steps, 2))      # Joint velocities (rad/s)
qdd = np.zeros((steps, 2))     # Joint accelerations (rad/s^2)

# Initial conditions
q[0] = [0.5, 0.5]
qd[0] = [0.0, 0.0]

In [ ]:
# ---------------------------------------
# Dynamics Function for the Planar 2R Arm
# ---------------------------------------
def dynamics(q, qd, tau):
    q1, q2 = q
    q1d, q2d = qd

# Mass matrix M(q)
    M11 = m1 * l1**2 + m2 * (l1**2 + 2 * l1 * l2 * np.cos(q2) + l2**2)
    M12 = m2 * (l1 * l2 * np.cos(q2) + l2**2)
    M21 = M12
    M22 = m2 * l2**2

    M = np.array([
        [M11, M12],
        [M21, M22]
    ])

# Vector of coriolis & centripetal torques
    h = m2 * l1 * l2 * np.sin(q2)

    C = np.array([
        -h * (2 * q1d * q2d + q2d**2) + 10 * q1d,
        h * q1d**2 + 0.01 * q2d
    ])

# Vector of gravitational torques
    G1 = ((m1 + m2) * l1 * g * np.cos(q1) +
          m2 * l2 * g * np.cos(q1 + q2))

    G2 = m2 * l2 * g * np.cos(q1 + q2)

    G = np.array([G1, G2])

# Compute joint accelerations
    qdd = np.linalg.inv(M) @ (tau - C - G)

    return qdd

In [ ]:
# ------------------------------------------
# Numerical Integration using Euler's Method
# ------------------------------------------
for i in range(steps - 1):
    qdd[i] = dynamics(q[i], qd[i], tau[i])

    qd[i + 1] = qd[i] + qdd[i] * dt
    q[i + 1] = q[i] + qd[i + 1] * dt

In [ ]:
# -----------------------------
# Forward Kinematics
# -----------------------------
def forward_kinematics(q):
    q1, q2 = q

    x1 = l1 * np.cos(q1)
    y1 = l1 * np.sin(q1)

    x2 = x1 + l2 * np.cos(q1 + q2)
    y2 = y1 + l2 * np.sin(q1 + q2)

    return (0, x1, x2), (0, y1, y2)

In [ ]:
# -----------------------------
# Visualization
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')
ax.grid(True)
ax.set_title('2-Link Robot Arm Torque Simulation')

line, = ax.plot([], [], 'o-', lw=4)
trace, = ax.plot([], [], '--', lw=1)

end_effector_x = []
end_effector_y = []

# Initialization function

def init():
    line.set_data([], [])
    trace.set_data([], [])
    return line, trace

# Animation update function
def update(frame):
    x, y = forward_kinematics(q[frame])

    line.set_data(x, y)

    end_effector_x.append(x[-1])
    end_effector_y.append(y[-1])

    trace.set_data(end_effector_x, end_effector_y)

    return line, trace

# Repeat continuously
ani = FuncAnimation.FuncAnimation(
    fig,
    update,
    frames=steps,
    init_func=init,
    interval=dt * 1000,
    blit=True,
    repeat=True
)
plt.close(fig)

In [ ]:
from IPython.display import HTML
import matplotlib.pyplot as plt

# Increase the animation embed limit to accommodate larger animations
plt.rcParams['animation.embed_limit'] = 50.0  # Set to 50 MB, adjust as needed

HTML(ani.to_jshtml())